<a href="https://colab.research.google.com/github/csbell-vu/in-class-notebooks/blob/main/SmolLM2_Pretraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploring Language Model Pre-Training

**DS 5690 · Week 8 Thursday · Pretraining from random initialization**

We will keep the SmolLM2 architecture recognizable, make it small enough to
train during class, and inspect what changes as it learns to predict the next
token in a collection of short stories.

## 1. Setup

In Colab, choose **Runtime → Change runtime type → T4 GPU** before running the
notebook.

In [ ]:
%pip install -q transformers datasets accelerate tensorboard

In [ ]:
import math

import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(42)

In [ ]:
torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU available"

## 2. Begin with the official SmolLM2 configuration

`AutoConfig` downloads the model's configuration, not its trained weights.
This lets us inspect the published architecture before changing it.

Source: [SmolLM2-135M configuration](https://huggingface.co/HuggingFaceTB/SmolLM2-135M/blob/main/config.json)

In [ ]:
model_name = "HuggingFaceTB/SmolLM2-135M"
config = AutoConfig.from_pretrained(model_name)
config

In [ ]:
# show config
config

In [ ]:
# creating a copy for comparison later
from copy import deepcopy
original_config = deepcopy(config)

## 3. Use the SmolLM2 tokenizer

We are changing the size of the transformer, but not its language. The model
will use the same vocabulary and token IDs as the official SmolLM2 family.

SmolLM2 has an end-of-text token but no padding token. We add a distinct pad
token so the model can still learn from the end-of-story tokens in our data.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
#tokenizer.add_special_tokens({"pad_token": "<|pad|>"})
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer

In [ ]:
text = "Once upon a time there was a little dragon who loved books."
token_ids = tokenizer(text, add_special_tokens=False).input_ids

print(tokenizer.convert_ids_to_tokens(token_ids))
print(token_ids)

## 4. Scale the configuration to classroom size

We keep the same Llama-style decoder family, RMS normalization, rotary
position embeddings, gated feed-forward layers, and grouped-query attention.
We reduce the repeated layers, representation width, feed-forward width, and
context length so the optimizer has a much smaller model to update.

We also tell the model about the padding token we just added.

In [ ]:
config.max_position_embeddings = 256
config.num_hidden_layers = 6
config.hidden_size = 384
config.num_attention_heads = 6
config.num_key_value_heads = 2
config.head_dim = 64
config.intermediate_size = 1024
config.use_cache = False
config.vocab_size = len(tokenizer)
config.pad_token_id = tokenizer.pad_token_id
config.bos_token_id = tokenizer.bos_token_id
config.eos_token_id = tokenizer.eos_token_id
config.dtype = torch.float32

config

Let's compare with the original:

In [ ]:
# configuration keys that we're setting
config_keys = ["max_position_embeddings", "hidden_size", "intermediate_size" ,"num_hidden_layers", "num_attention_heads", "num_key_value_heads", "head_dim", "use_cache", "vocab_size", "pad_token_id", "bos_token_id", "eos_token_id",
]

config_compare = pd.DataFrame({'original':original_config.to_dict(), 'new': config.to_dict()})
config_compare.loc[config_keys, :]

Check the two relationships that make the attention dimensions work. Each
query head receives 64 hidden features, and each key/value head is shared by
three query heads.

In [ ]:
head_dimension = config.hidden_size / config.num_attention_heads
queries_per_kv_head = config.num_attention_heads / config.num_key_value_heads

print("Features per attention head:", head_dimension)
print("Query heads per key/value head:", queries_per_kv_head)

## 5. Construct the model from random weights

`from_config` builds the architecture described by `config`. It does not load
the pretrained SmolLM2 parameters.

In [ ]:
model = AutoModelForCausalLM.from_config(config)
model

In [ ]:
model.model.layers[0]

In [ ]:
first_attention = model.model.layers[0].self_attn

print("Query projection: ", tuple(first_attention.q_proj.weight.shape))
print("Key projection:   ", tuple(first_attention.k_proj.weight.shape))
print("Value projection: ", tuple(first_attention.v_proj.weight.shape))
print("Output projection:", tuple(first_attention.o_proj.weight.shape))

In [ ]:
parameter_count = sum(parameter.numel() for parameter in model.parameters())
f"{parameter_count:,} parameters ({parameter_count / 1_000_000:.1f}M)"

## 6. Download TinyStories

TinyStories contains short synthetic stories written with a deliberately small
vocabulary. The `10M` configuration contains about 10 million words and about
62,000 stories, with train and validation splits already defined.

Sources: [TinyStories paper](https://arxiv.org/abs/2305.07759) and
[TinyStories 10M dataset](https://huggingface.co/datasets/eminorhan/tinystories)

In [ ]:
stories = load_dataset("roneneldan/tinystories")
stories

In [ ]:
stories["train"][0]

In [ ]:
sample_text_tokenized = tokenizer.tokenize(stories["train"][0]["text"])

# Get length of tokenized text
len(sample_text_tokenized)

## 7. Tokenize and pack the stories

A story can be shorter or longer than one context window. The model does not
need one row per story; it needs a continuous stream of token IDs divided into
256-token contexts.

Within each mapping batch, we join stories with the end-of-text token and ask
the tokenizer to cut that stream into consecutive contexts. This allows one
context to contain the end of one story and the beginning of the next.

Most chunks are full. We keep the final, shorter chunk from each mapping batch
rather than throwing those tokens away. The data collator will pad those few
chunks when it assembles a training batch.

In [ ]:
context_length = config.max_position_embeddings


def tokenize(batch):
    text = tokenizer.eos_token.join(batch["text"]) + tokenizer.eos_token
    return tokenizer(
        text,
        add_special_tokens=False,
        truncation=True,
        max_length=context_length,
        return_overflowing_tokens=True,
        return_length=True,
    )

In [ ]:
packed_stories = stories.map(
    tokenize,
    batched=True,
    batch_size=1000,
    num_proc = 10,
    remove_columns=stories["train"].column_names,
)

packed_stories

Let's make sure this looks correct. Nearly every row should
be a complete context, and decoding one row should reveal end-of-story
boundaries inside the token stream.

In [ ]:
pd.Series(packed_stories["train"]["length"]).value_counts().sort_index().tail()

In [ ]:
tokenizer.decode(packed_stories["train"][0]["input_ids"])

In [ ]:
packed_stories = packed_stories.remove_columns(
    ["attention_mask", "length", "overflow_to_sample_mapping"]
)

## 8. Assemble causal-language-modeling batches

With `mlm=False`, the collator prepares labels for next-token prediction. It
also pads the short remainder chunks and replaces their pad labels with
`-100`, which tells PyTorch not to score those positions.

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

batch = data_collator([packed_stories["train"][0], packed_stories["train"][-1]])
batch

In [ ]:
batch["input_ids"].shape, batch["labels"].shape

For causal language modeling, the input IDs are also the labels. The model
shifts them internally so that the representation at one position predicts
the token at the next position. Pad labels are `-100`, so they are not scored.

In [ ]:
batch["input_ids"][0, :12], batch["labels"][0, :12]

## 9. Decide what to observe

Validation loss asks, "How surprised is the model by held-out text?"
Perplexity expresses that same uncertainty on an exponential scale.
Next-token accuracy asks the more concrete question, "How often was the
model's highest-scoring next token correct?"

In [ ]:
def keep_metrics(logits, labels):
    logits = logits[:, :-1].float()
    labels = labels[:, 1:]
    losses = torch.nn.functional.cross_entropy(
        logits.flatten(0, 1),
        labels.flatten(),
        reduction="none",
        ignore_index=-100,
    ).view_as(labels)
    return logits.argmax(dim=-1), losses


def compute_metrics(eval_prediction):
    (predictions, losses), labels = eval_prediction
    labels = labels[:, 1:]
    scored = labels != -100
    return {
        "token_accuracy": (predictions[scored] == labels[scored]).mean(),
        "perplexity": math.exp(losses[scored].mean()),
    }

We will also return to the same prompts after each epoch. Greedy decoding
removes sampling variation, so changes come from the model's changing weights.

In [ ]:
prompts = [
    "Once upon a time",
    "The little girl opened the door and",
    "The dragon was afraid because",
]

In [ ]:
def complete(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        use_cache=True,
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

## 10. Configure the training run

These are the controls worth changing for this experiment: batch size,
gradient accumulation, learning rate, checkpointing, precision, logging, and
the output location. Everything else keeps the library default.

In [ ]:
output_dir = "smollm2-tinystories"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    gradient_checkpointing=False,
    learning_rate=5e-4,
    weight_decay=0.1,
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),
    logging_steps=25,
    save_strategy="epoch",
    save_total_limit=3,
    report_to="tensorboard",
    #log_dir=f"{output_dir}/runs",
)

training_args

We use a bounded validation sample because frequent feedback matters more here
than computing the most precise possible estimate.

In [ ]:
eval_stories = packed_stories["validation"].select(
    range(min(512, len(packed_stories["validation"])))
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=packed_stories["train"],
    eval_dataset=eval_stories,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=keep_metrics,
)

trainer

In [ ]:
[complete(prompt) for prompt in prompts]

At random initialization, the output should not resemble a sensible story.
The architecture supplies the computation; training data and parameter
updates produce the behavior.

TensorBoard runs locally inside Colab and does not require an account. Open it
before training. It will update as `Trainer` writes new loss records, and the
evaluation cells will add one validation point after each epoch.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir smollm2-tinystories/runs

## 11. Train, inspect, and continue

The first evaluation measures random initialization. Each training cell then
occupies the GPU for one epoch. Compare the validation metrics, TensorBoard
curves, and completions before deciding whether to continue.

In [ ]:
baseline = trainer.evaluate()
baseline

In [ ]:
model.float()

In [ ]:
trainer.train()

In [ ]:
evaluation = trainer.evaluate()
evaluation

In [ ]:
[complete(prompt) for prompt in prompts]

`Trainer` keeps the numeric history behind the TensorBoard curves in
`state.log_history`. Its checkpoint contains the model plus the optimizer,
learning-rate scheduler, and random state. Increase the total number of epochs
and resume from the latest checkpoint to continue the same run.

In [ ]:
pd.DataFrame(trainer.state.log_history).tail()

In [ ]:
trainer.args.num_train_epochs = 2
trainer.train(resume_from_checkpoint=True)

In [ ]:
evaluation = trainer.evaluate()
evaluation

In [ ]:
[complete(prompt) for prompt in prompts]

One more epoch gives us a third observation point. On a slower Colab GPU, this
cell can be skipped or run after class.

In [ ]:
trainer.args.num_train_epochs = 3
trainer.train(resume_from_checkpoint=True)

In [ ]:
evaluation = trainer.evaluate()
evaluation

In [ ]:
[complete(prompt) for prompt in prompts]

## 12. Save the final model

In [ ]:
trainer.save_model(f"{output_dir}/final")
tokenizer.save_pretrained(f"{output_dir}/final")

## Closing interpretation

We held the architecture and prompts constant while optimization changed the
weights. The loss curves summarize millions of next-token comparisons. The
completions translate some of that numeric change back into behavior we can
read.

This run does not produce a general-purpose base model. It gives us a
controlled observation of pretraining. In a later notebook, we can load the
official SmolLM2 base checkpoint and ask what extensive pretraining makes
possible before supervised fine-tuning or preference optimization begins.

Compare the three kinds of evidence from this run:

- Did training and validation loss move together?
- How much did strict next-token accuracy change?
- When did the completions first show word structure, grammar, or story-like
  behavior?
- What still appears to require more data or more optimization?